# Semi-Supervised Graph Learning & Active Learning Visualizations

Welcome to the demonstration suite for the `graphalp` package. This notebook contains the implementation to generate representative graphs, execute label propagation and active learning samplers, and visualize their step-by-step convergence as animated GIFs.

We use a **Stochastic Block Model (SBM)** graph which perfectly models a community-split graph structure with a narrow bottleneck bridge. This helps illustrate how information flows during label propagation and how active learning samplers identify informative boundaries.

### Visualized Algorithms
1. **Label Propagation**
   * **Harmonic LP (Dirichlet Problem)**: Jacobi iterative convergence showing probability diffusion.
   * **Min-Cut LP**: Network flow source-sink cuts separating classes.
   * **Spectral LP**: Dimensionality reduction using Laplacian eigenvectors + SVM boundaries.
2. **Active Learning Samplers**
   * **Random Sampler**: Uniform baseline queries.
   * **S2 Sampler**: Path bisection search along labeled boundaries.
   * **Harmonic Greedy Sampler**: Expected risk minimization querying the most uncertain nodes.

In [ ]:
import sys
import os
import io
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

# Ensure local src/ folder is imported correctly
sys.path.insert(0, os.path.abspath('../src'))

from graphalp.label_propagation import HarmonicLabelPropagator, MinCutLabelPropagator, SpectralLabelPropagator
from graphalp.active_learning import HarmonicGreedySampler, S2Sampler, RandomSampler
from graphalp.utils import compute_risk

## 1. Setup Theme & SBM Graph Generation

We define a custom dark theme matching professional dark mode specs, and build a 30-node connected SBM graph with a fixed spring layout.

In [ ]:
# Theme HSL-based colors
BG_COLOR = '#0f172a'
EDGE_COLOR = '#334155'
NODE_BLUE = '#2563eb'
BORDER_BLUE = '#1d4ed8'
NODE_RED = '#e11d48'
BORDER_RED = '#be123c'
NODE_GREY = '#64748b'
BORDER_GREY = '#475569'
HIGHLIGHT_GOLD = '#f59e0b'
WHITE = '#f8fafc'

def get_predicted_color(p):
    r = int(37 + (225 - 37) * p)
    g = int(99 + (29 - 99) * p)
    b = int(235 + (72 - 235) * p)
    return f'#{r:02x}{g:02x}{b:02x}'

def fig_to_img(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', facecolor=fig.get_facecolor(), edgecolor='none', dpi=100)
    buf.seek(0)
    img = Image.open(buf)
    img.load()
    buf.close()
    return img

def draw_base_graph(ax, G, pos, labels, predictions, highlight_node=None, title=""):
    ax.set_facecolor(BG_COLOR)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=EDGE_COLOR, width=1.0, alpha=0.5)
    for node in G.nodes():
        label = labels.get(node)
        pred = predictions.get(node, 0.5)
        if label == 0:
            color, border, width, size = NODE_BLUE, BORDER_BLUE, 3.0, 500
        elif label == 1:
            color, border, width, size = NODE_RED, BORDER_RED, 3.0, 500
        else:
            color, border, width, size = get_predicted_color(pred), BORDER_GREY, 1.0, 350
        nx.draw_networkx_nodes(G, pos, nodelist=[node], node_color=color, node_size=size,
                               edgecolors=border, linewidths=width, ax=ax)
    if highlight_node is not None:
        nx.draw_networkx_nodes(G, pos, nodelist=[highlight_node], node_color='none', node_size=750,
                               edgecolors=HIGHLIGHT_GOLD, linewidths=3.0, ax=ax)
        x, y = pos[highlight_node]
        ax.text(x, y + 0.12, "Query!", color=HIGHLIGHT_GOLD, fontsize=11, fontweight='bold',
                horizontalalignment='center', bbox=dict(facecolor=BG_COLOR, edgecolor=HIGHLIGHT_GOLD, boxstyle='round,pad=0.2'))
    ax.set_title(title, color=WHITE, fontsize=14, fontweight='bold', pad=15)
    ax.axis('off')

# SBM Generation
np.random.seed(42)
sizes = [15, 15]
probs = [[0.35, 0.03], [0.03, 0.35]]
G = nx.stochastic_block_model(sizes, probs, seed=42)

if not nx.is_connected(G):
    components = list(nx.connected_components(G))
    while len(components) > 1:
        u, v = list(components[0])[0], list(components[1])[0]
        G.add_edge(u, v)
        components = list(nx.connected_components(G))

pos = nx.spring_layout(G, k=0.4, iterations=100, seed=42)
gt_labels = {n: 0 if n < 15 else 1 for n in G.nodes()}

fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, {0:0, 29:1}, {}, title="Starting SBM Graph Structure")
plt.show()

## 2. Harmonic Label Propagation - Jacobi Iteration Animation

We show the Jacobi solver running step-by-step, allowing us to watch label probabilities diffuse organically outwards.

In [ ]:
print("Generating Harmonic LP...")
frames = []
labels = {0: 0, 29: 1}
f_iter = {n: 0.5 for n in G.nodes()}
f_iter[0] = 0.0
f_iter[29] = 1.0

for step in range(21):
    fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
    title = f"Harmonic Label Propagation (Jacobi Iteration {step})"
    draw_base_graph(ax, G, pos, labels, f_iter, title=title)
    plt.tight_layout()
    frames.append(fig_to_img(fig))
    plt.close(fig)
    
    f_next = f_iter.copy()
    for u in G.nodes():
        if u not in labels:
            neighbors = list(G.neighbors(u))
            f_next[u] = sum(f_iter[v] for v in neighbors) / len(neighbors)
    f_iter = f_next

os.makedirs('../docs/gifs', exist_ok=True)
frames += [frames[-1]] * 15
frames[0].save('../docs/gifs/harmonic_propagation.gif', save_all=True, append_images=frames[1:], duration=350, loop=0)
print("Saved docs/gifs/harmonic_propagation.gif")

## 3. Min-Cut Label Propagation Animation

We show the minimum cut algorithm adding source/sink edges, discovering the mincut wall, and creating hard bipartitions.

In [ ]:
print("Generating Min-Cut LP...")
frames = []
labels = {0: 0, 29: 1}

# Frame 0: Initial
fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, labels, {}, title="Min-Cut: Starting Seed Labels")
plt.tight_layout()
f0 = fig_to_img(fig)
plt.close(fig)
frames += [f0] * 12

# Frame 1: Source & Sink
fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, labels, {}, title="Min-Cut: Connect to Virtual Source & Sink")
src_pos, snk_pos = (-1.5, 0.0), (1.5, 0.0)
ax.scatter(src_pos[0], src_pos[1], color='#10b981', s=600, marker='s', edgecolors='#047857', zorder=5)
ax.text(src_pos[0], src_pos[1]-0.15, "Source (s)", color='#10b981', fontweight='bold', horizontalalignment='center')
ax.scatter(snk_pos[0], snk_pos[1], color='#8b5cf6', s=600, marker='s', edgecolors='#6d28d9', zorder=5)
ax.text(snk_pos[0], snk_pos[1]-0.15, "Sink (t)", color='#8b5cf6', fontweight='bold', horizontalalignment='center')
ax.plot([src_pos[0], pos[0][0]], [src_pos[1], pos[0][1]], color='#10b981', linestyle='--', linewidth=2.0)
ax.plot([snk_pos[0], pos[29][0]], [snk_pos[1], pos[29][1]], color='#8b5cf6', linestyle='--', linewidth=2.0)
plt.tight_layout()
f1 = fig_to_img(fig)
plt.close(fig)
frames += [f1] * 12

# Frame 2: MinCut Wall
propagator = MinCutLabelPropagator(G)
propagator.fit([0, 29], [0, 1])
cut_edges = propagator.mincut_wall
fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, labels, {}, title="Min-Cut: Compute Minimum s-t Cut")
nx.draw_networkx_edges(G, pos, ax=ax, edgelist=cut_edges, edge_color='#f59e0b', width=3.5, style='dashed')
plt.tight_layout()
f2 = fig_to_img(fig)
plt.close(fig)
frames += [f2] * 15

# Frame 3: Final Partition
predictions = {n: 1 if n in propagator.reachable else 0 for n in G.nodes()}
fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, {}, predictions, title="Min-Cut: Final Hard Class Bipartition")
nx.draw_networkx_edges(G, pos, ax=ax, edgelist=cut_edges, edge_color='#f59e0b', width=2.0, style='dotted', alpha=0.7)
plt.tight_layout()
f3 = fig_to_img(fig)
plt.close(fig)
frames += [f3] * 20

frames[0].save('../docs/gifs/mincut_propagation.gif', save_all=True, append_images=frames[1:], duration=150, loop=0)
print("Saved docs/gifs/mincut_propagation.gif")

## 4. Spectral Label Propagation Animation

We show the mapping of nodes to the 2D Laplacian eigenvector space, the fit of the SVM decision boundary, and the back-projected probabilities.

In [ ]:
print("Generating Spectral LP...")
frames = []
labels = {0: 0, 29: 1}

# Frame 0: Physical
fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, labels, {}, title="Spectral LP: Physical Graph")
plt.tight_layout()
s0 = fig_to_img(fig)
plt.close(fig)
frames += [s0] * 12

# Compute Spectral
spectral_prop = SpectralLabelPropagator(G, n_components=3)
spectral_prop.fit([0, 29], [0, 1])
embeddings = spectral_prop.embeddings[:, 1:3]

# Frame 1: Embedding Space Scatter
fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
ax.set_facecolor(BG_COLOR)
ax.spines['bottom'].set_color('#334155')
ax.spines['left'].set_color('#334155')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(colors=WHITE)
for n in G.nodes():
    emb = embeddings[n]
    color, size = (NODE_BLUE, 180) if n==0 else ((NODE_RED, 180) if n==29 else (NODE_GREY, 80))
    ax.scatter(emb[0], emb[1], color=color, s=size, edgecolors=WHITE if n in [0, 29] else '#475569', zorder=5)
ax.set_title("Spectral LP: Nodes in 2D Eigenvector Space", color=WHITE, fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Eigenvector 1", color=WHITE)
ax.set_ylabel("Eigenvector 2", color=WHITE)
plt.tight_layout()
s1 = fig_to_img(fig)
plt.close(fig)
frames += [s1] * 12

# Frame 2: SVM Boundaries
fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
ax.set_facecolor(BG_COLOR)
ax.spines['bottom'].set_color('#334155')
ax.spines['left'].set_color('#334155')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(colors=WHITE)

x_min, x_max = embeddings[:, 0].min() - 0.05, embeddings[:, 0].max() + 0.05
y_min, y_max = embeddings[:, 1].min() - 0.05, embeddings[:, 1].max() + 0.05
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# Prepend constant first column so it matches SVC expected 3D features
const_val = spectral_prop.embeddings[0, 0]
grid_points_3d = np.c_[np.full(grid_points.shape[0], const_val), grid_points]
Z = spectral_prop.svc.predict_proba(grid_points_3d)[:, 1].reshape(xx.shape)

colors = [(37/255, 99/255, 235/255), (15/255, 23/255, 42/255), (225/255, 29/255, 72/255)]
cmap_spectral = LinearSegmentedColormap.from_list('blue_red_spectral', colors, N=100)

ax.contourf(xx, yy, Z, levels=50, cmap=cmap_spectral, alpha=0.4, zorder=1)
ax.contour(xx, yy, Z, levels=[0.5], colors=HIGHLIGHT_GOLD, linewidths=2.0, linestyles='dashed', zorder=2)
for n in G.nodes():
    emb = embeddings[n]
    color, size = (NODE_BLUE, 180) if n==0 else ((NODE_RED, 180) if n==29 else (NODE_GREY, 80))
    ax.scatter(emb[0], emb[1], color=color, s=size, edgecolors=WHITE if n in [0, 29] else '#475569', zorder=5)
ax.set_title("Spectral LP: Fit SVM in Embedding Space", color=WHITE, fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Eigenvector 1", color=WHITE)
ax.set_ylabel("Eigenvector 2", color=WHITE)
plt.tight_layout()
s2 = fig_to_img(fig)
plt.close(fig)
frames += [s2] * 15

# Frame 3: Physical predictions
preds_spectral = dict(zip(G.nodes(), spectral_prop.predict_probabilities(list(G.nodes()))))
fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, {}, preds_spectral, title="Spectral LP: Predicted Probabilities on Graph")
plt.tight_layout()
s3 = fig_to_img(fig)
plt.close(fig)
frames += [s3] * 20

frames[0].save('../docs/gifs/spectral_propagation.gif', save_all=True, append_images=frames[1:], duration=150, loop=0)
print("Saved docs/gifs/spectral_propagation.gif")

## 5. Active Learning Sampler Comparison Loop

We run the Random, S2, and Harmonic Greedy active learning loops for 10 queries, saving visual GIFs and plotting risk reduction comparison curves.

In [ ]:
sampler_classes = {
    'harmonic_greedy': (HarmonicGreedySampler, "Harmonic Greedy Sampler"),
    's2': (lambda G: S2Sampler(G, fallback_sampler=RandomSampler(G)), "S2 Sampler"),
    'random': (RandomSampler, "Random Sampler")
}
all_sampler_risks = {}

for key, (cls_init, name) in sampler_classes.items():
    print(f"Running {name}...")
    frames = []
    np.random.seed(42)
    X_labeled, y_labeled = [0, 29], [0, 1]
    sampler = cls_init(G)
    sampler.fit(X_labeled, y_labeled)
    
    prop = HarmonicLabelPropagator(G)
    prop.fit(X_labeled, y_labeled)
    all_preds = dict(zip(G.nodes(), prop.predict_probabilities(list(G.nodes()))))
    initial_risk = compute_risk(np.array([prop.f[n] for n in prop.nodes if n not in prop.labels]))
    risks = [initial_risk]
    
    fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
    draw_base_graph(ax, G, pos, dict(zip(X_labeled, y_labeled)), all_preds, title=f"{name}: Initial (Risk: {initial_risk:.2f})")
    plt.tight_layout()
    frames.append(fig_to_img(fig))
    plt.close(fig)
    
    for step in range(1, 11):
        next_node = sampler.sample()
        if next_node is None: break
        label = gt_labels[next_node]
        
        fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
        draw_base_graph(ax, G, pos, dict(zip(X_labeled, y_labeled)), all_preds, highlight_node=next_node, title=f"{name}: Query Node {next_node} (Step {step})")
        plt.tight_layout()
        frames.append(fig_to_img(fig))
        plt.close(fig)
        
        X_labeled.append(next_node)
        y_labeled.append(label)
        sampler.fit(X_labeled, y_labeled)
        
        prop.fit(X_labeled, y_labeled)
        all_preds = dict(zip(G.nodes(), prop.predict_probabilities(list(G.nodes()))))
        current_risk = compute_risk(np.array([prop.f[n] for n in prop.nodes if n not in prop.labels]))
        risks.append(current_risk)
        
        fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG_COLOR)
        draw_base_graph(ax, G, pos, dict(zip(X_labeled, y_labeled)), all_preds, title=f"{name}: Post-Query (Risk: {current_risk:.2f})")
        plt.tight_layout()
        frames.append(fig_to_img(fig))
        plt.close(fig)
        
    all_sampler_risks[key] = risks
    frames += [frames[-1]] * 10
    frames[0].save(f'../docs/gifs/{key}_sampler.gif', save_all=True, append_images=frames[1:], duration=600, loop=0)
    print(f"Saved docs/gifs/{key}_sampler.gif")

# Comparison Plot
fig, ax = plt.subplots(figsize=(8, 5), facecolor=BG_COLOR)
ax.set_facecolor(BG_COLOR)
ax.spines['bottom'].set_color('#334155')
ax.spines['left'].set_color('#334155')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(colors=WHITE)
steps = np.arange(0, 11)

ax.plot(steps, all_sampler_risks['harmonic_greedy'], color='#f59e0b', marker='o', linewidth=2.5, label='Harmonic Greedy')
ax.plot(steps, all_sampler_risks['s2'], color='#8b5cf6', marker='s', linewidth=2.0, label='S2 Sampler')
ax.plot(steps, all_sampler_risks['random'], color='#64748b', marker='^', linewidth=1.5, linestyle='--', label='Random Sampler')

ax.set_title("Active Learning Performance Comparison", color=WHITE, fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Query Step", color=WHITE)
ax.set_ylabel("Expected Graph Uncertainty (Risk)", color=WHITE)
ax.legend(facecolor=BG_COLOR, edgecolor='#334155', labelcolor=WHITE)
ax.grid(color='#334155', linestyle=':', linewidth=0.5)
plt.tight_layout()
fig.savefig('../docs/images/risk_comparison.png', facecolor=fig.get_facecolor(), dpi=150)
plt.show()